In [0]:
%run ../gold/_shared/watermark

In [0]:
%run ../gold/_shared/cdf_reader

In [0]:
%run ../gold/_shared/dim_lookup

In [0]:
# 1. watermark on a fact that's never run → -1
print("watermark fact_sales_line:", get_watermark("fact_sales_line"))   # expect -1

# 2. current_version on a silver table → some int
print("sa_tran_item version:", current_version("retaildp.silver.sa_tran_item"))

# 3. dim_lookup smoke — attach all keys to a few silver item rows
from pyspark.sql import functions as F
sample = spark.table("retaildp.silver.sa_tran_item").limit(100)
sample = add_date_key(sample)
sample = add_store_key(sample)
sample = add_channel_key(sample)
sample = add_item_key(sample)
display(sample.select("ITEM","STORE","BUSINESS_DATE","RTLOG_ORIG_SYS",
                      "date_key","store_key","channel_key","item_key").limit(20))

# 4. orphan check — any -1 keys? (should be none for valid silver rows)
for k in ["date_key","store_key","channel_key","item_key"]:
    n = sample.where(F.col(k) == -1).count()
    print(f"{k} = -1 count:", n)